In [1]:
print('wowza')

wowza


In [1]:
import os
import kagglehub
import pandas as pd

# Returns the local cache path (~/.cache/kagglehub/...) without re-downloading
path = kagglehub.dataset_download("uciml/german-credit")

# First column of the CSV is an unnamed row index
df = pd.read_csv(os.path.join(path, "german_credit_data.csv"), index_col=0)

print(df.shape)
df.head()

/Users/shaamiazeem/venvs/mot/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/shaamiazeem/venvs/mot/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(1000, 9)


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,67,male,2,own,NaN,little,1169,6,radio/TV
1,22,female,2,own,little,moderate,5951,48,radio/TV
2,49,male,1,own,little,NaN,2096,12,education
3,45,male,2,free,little,little,7882,42,furniture/equipment
4,53,male,2,free,little,little,4870,24,car


In [2]:
df.head(5)

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,67,male,2,own,NaN,little,1169,6,radio/TV
1,22,female,2,own,little,moderate,5951,48,radio/TV
2,49,male,1,own,little,NaN,2096,12,education
3,45,male,2,free,little,little,7882,42,furniture/equipment
4,53,male,2,free,little,little,4870,24,car


In [3]:
import sqlite3

# Snake_case column names so you don't have to quote them in SQL
# ("Credit amount" becomes credit_amount)
sql_df = df.copy()
sql_df.columns = sql_df.columns.str.strip().str.lower().str.replace(" ", "_")

with sqlite3.connect("german_credit.db") as conn:
    sql_df.to_sql("german_credit", conn, if_exists="replace", index=False)

    # Quick check that the table was written
    print(pd.read_sql("SELECT COUNT(*) AS n_rows FROM german_credit", conn))
    print(pd.read_sql("SELECT * FROM german_credit LIMIT 5", conn))


   n_rows
0    1000
   age     sex  job housing saving_accounts checking_account  credit_amount  \
0   67    male    2     own            None           little           1169   
1   22  female    2     own          little         moderate           5951   
2   49    male    1     own          little             None           2096   
3   45    male    2    free          little           little           7882   
4   53    male    2    free          little           little           4870   

   duration              purpose  
0         6             radio/TV  
1        48             radio/TV  
2        12            education  
3        42  furniture/equipment  
4        24                  car  


In [4]:
from sklearn.linear_model import LogisticRegression
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Age               1000 non-null   int64 
 1   Sex               1000 non-null   object
 2   Job               1000 non-null   int64 
 3   Housing           1000 non-null   object
 4   Saving accounts   817 non-null    object
 5   Checking account  606 non-null    object
 6   Credit amount     1000 non-null   int64 
 7   Duration          1000 non-null   int64 
 8   Purpose           1000 non-null   object
dtypes: int64(4), object(5)
memory usage: 78.1+ KB


In [5]:
# missing values occur in the savings and checkings accounts, is this random?
na = df.isna()
na

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,False,False,False,False,True,False,False,False,False
1,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,True,False,False,False
3,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
995,False,False,False,False,False,True,False,False,False
996,False,False,False,False,False,False,False,False,False
997,False,False,False,False,False,True,False,False,False
998,False,False,False,False,False,False,False,False,False


In [6]:
saving_na=df[na["Saving accounts"]== True]
saving_na

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,67,male,2,own,NaN,little,1169,6,radio/TV
5,35,male,1,free,NaN,NaN,9055,36,education
16,53,male,2,own,NaN,NaN,2424,24,radio/TV
17,25,male,2,own,NaN,little,8072,30,business
24,26,male,2,own,NaN,NaN,2069,10,furniture/equipment
...,...,...,...,...,...,...,...,...,...
968,29,male,2,rent,NaN,NaN,7166,42,radio/TV
977,42,male,2,own,NaN,moderate,2427,18,business
990,37,male,1,own,NaN,NaN,3565,12,education
992,23,male,1,rent,NaN,little,1936,18,radio/TV


In [7]:
check_na=df[na["Checking account"] == True]
check_na

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
2,49,male,1,own,little,NaN,2096,12,education
5,35,male,1,free,NaN,NaN,9055,36,education
6,53,male,2,own,quite rich,NaN,2835,24,furniture/equipment
8,61,male,1,own,rich,NaN,3059,12,radio/TV
16,53,male,2,own,NaN,NaN,2424,24,radio/TV
...,...,...,...,...,...,...,...,...,...
990,37,male,1,own,NaN,NaN,3565,12,education
991,34,male,1,own,moderate,NaN,1569,15,radio/TV
994,50,male,2,own,NaN,NaN,2390,12,car
995,31,female,1,own,little,NaN,1736,12,furniture/equipment


In [9]:
# 1 means a good credit score, 2 means a bad credit score
#Job (numeric: 0 - unskilled and non-resident, 1 - unskilled and resident, 2 - skilled, 3 - highly skilled)
check_na['Credit Risk'].value_counts()

KeyError: 'Credit Risk'